In [1]:
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http import models
import re
import typing as tp
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi


/mnt/shared/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
COLLECTION_NAME = "freecad_docs"
DATA_PATH = "data"
DOCS_PATH = os.path.join(DATA_PATH, "docs")
EMBEDDINGS_PATH = "embeddings"

# Metrics

In [3]:
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

In [4]:
def mean_average_cosine_similarity(similarities: tp.List[tp.List[float]]) -> float:
    return sum(sum(sims) / len(sims) for sims in similarities) / len(similarities)

In [5]:
def mean_max_cosine_similarity(similarities: tp.List[tp.List[float]]) -> float:
    return sum(max(sims) for sims in similarities) / len(similarities)

In [6]:
def hit_rate_at_k(similarities: tp.List[tp.List[float]], threshold: float, k: int) -> float:
    hits = 0
    for sims in similarities:
        top_k = sims[:k]
        if any(sim >= threshold for sim in top_k):
            hits += 1
    return hits / len(similarities)

# Qdrant init

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [8]:
embedding_model = SentenceTransformer("BAAI/bge-m3", device=device)

In [9]:
client = QdrantClient(path="qdrant")

In [10]:
def filter_code(doc: str) -> str:
    code_blocks = re.findall(r"```(?:[a-zA-Z]+\n)?(.*?)```", doc, re.DOTALL)
    code = "\n\n".join(code_blocks)
    return re.sub(r'\n{3,}', '\n\n', code)

In [11]:
query_df = pd.read_csv(os.path.join(DATA_PATH, "tooltips_dataset.csv"))

# BM25 Baseline

In [97]:
for doc in corpus:
    print(hf_tokenizer.tokenize(doc))
    break

['▁Description', '▁It', '▁will', '▁serve', '▁to', '▁generate', '▁flat', ',', '▁shape', '-', 'based', '▁views', '▁from', '▁a', '▁Mes', 'h', '▁based', '▁object', ',', '▁to', '▁be', '▁used', '▁by', '▁the', '▁Arch', '▁Equip', 'ment', '▁tool', '.', '▁Usa', 'ge', '▁Select', '▁a', '▁Mes', 'h', '▁object', '.', '▁Select', '▁the', '▁button', ',', '▁or', '▁Arch', '▁→', '▁Utili', 'ties', '▁→', '▁3', 'View', 's', '▁from', '▁the', '▁top', '▁menu', '.', '▁Script', 'ing', '▁Arch', '▁API', '▁and', '▁Free', 'CAD', '▁Script', 'ing', '▁Basic', 's', '.', '▁This', '▁tool', '▁can', '▁be', '▁used', '▁in', '▁macro', 's', '▁and', '▁from', '▁the', '▁Python', '▁console', '▁by', '▁using', '▁the', '▁following', '▁function', ':', '▁``', '`', 'start', '_', 'code', '▁shape', '▁=', '▁create', 'M', 'esh', 'View', '(', 'ob', 'j', ',', '▁direction', '=', 'Free', 'CAD', '.', 'Ve', 'ctor', '(', '0', ',', '▁0', ',', '▁', '-1)', ',', '▁out', 'eron', 'ly', '=', 'Fa', 'lse', ',', '▁largest', 'on', 'ly', '=', 'Fa', 'lse', ')', '

In [120]:
query_idx = 2

In [125]:
docs_df = pd.read_csv(os.path.join(DATA_PATH, "chunks_df.csv"))
corpus = docs_df["content"].values

tokenized_corpus = []
for doc in corpus:
    # tokenized_corpus.append(word_tokenize(filter_code(str(doc))))
    tokenized_corpus.append(word_tokenize(str(doc)))


bm25 = BM25Okapi(tokenized_corpus)

def bm25_retrieve(query: str, k: int = 5):
    tokenized_query = query.split()
    scores = bm25.get_scores(tokenized_query)
    top_k_idx = np.argsort(scores)[::-1][:k]
    return [corpus[i] for i in top_k_idx], [scores[i] for i in top_k_idx]

In [122]:
def bm25_retrieve_with_score(queries: np.ndarray, y: np.ndarray, k: int = 5):
    retrieved_texts = []
    scores = []

    for i in range(len(queries)):
        results, _ = bm25_retrieve(queries[i], k=k)
        retrieved_texts.append(results)

        retrieved_vecs = embedding_model.encode([filter_code(text) for text in results])
        y_vec = embedding_model.encode(y[i])

        scores_batch = [cosine_similarity(vec, y_vec) for vec in retrieved_vecs]
        scores.append(scores_batch)

    return retrieved_texts, scores

In [ ]:
query = query_df.iloc[query_idx]["X"]
similarities, scores = bm25_retrieve(query)
print(query)
for i, (text, score) in enumerate(zip(similarities, scores)):
    print(f"=================Hit {i+1} score: {score} =================")

    print(f"{text}\n")

Add a window
=================Hit 1 score: 9.51023430853718 =================
Description

Usage

Options

Limitations

Scripting

```start_code

Add python code here

end_code```

=================Hit 2 score: 7.499336807575432 =================
```start_code

Add ppa:freecad-community/ppa to your software sources
sudo apt-get update
sudo apt-get install freecad-extras-assembly2

end_code```

In Windows
 download the git repository as ZIP
 assuming FreeCAD is installed in "C:\PortableApps\FreeCAD 0_15", go to "C:\PortableApps\FreeCAD 0_15\Mod" within Windows Explorer
 create new directory named "assembly2"
 unzip downloaded repository in "C:\PortableApps\FreeCAD 0_15\Mod\assembly2"

FreeCAD will now have a new workbench-entry called "Assembly 2".

Pyside and Numpy are integrated in the FreeCAD 0.15 dev-Snapshots, so these Python packages do not need to be installed individually

To update to the latest version, delete the assembly2 folder and redownload the git repository.

Links

 Wo

# Experiments

In [21]:
query_idx = 0

query = query_df.iloc[query_idx]["X"]
query_qdrant = query + " python code"
# query_qdrant = "Add window"
# query_qdrant = "Make a circle"


query_vec = embedding_model.encode(query_qdrant)
hits = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vec,
    limit=5,
)

for _ in range(len(hits.points)):
    print(hits.points[_].payload['source'])

retrieved_text_batch = [point.payload['content'] for point in hits.points]
retrieved_code_batch = [filter_code(text) for text in retrieved_text_batch]

retrieved_vec = embedding_model.encode(retrieved_code_batch)

y_vec = embedding_model.encode(query_df.iloc[query_idx]["Y"])

print(f"X: {query_df.iloc[query_idx]['X']}")
print("Y\n", query_df.iloc[query_idx]["Y"], "\n")
for i, vec in enumerate(retrieved_vec):
    print(f"Hit {i+1}, score: {cosine_similarity(vec, y_vec)}")

for i, text in enumerate(retrieved_text_batch):
    print(f"=================Hit {i+1}=================")
    print(text)


Arch_Wall.wikitext
Arch_CurtainWall.wikitext
Arch_CutPlane.wikitext
Arch_CurtainWall.wikitext
Arch_3Views.wikitext
X: Create a wall python code
Y
 import Arch
obj = Arch.makeWall() 

Hit 1, score: 0.6794925928115845
Hit 2, score: 0.6343596577644348
Hit 3, score: 0.6615268588066101
Hit 4, score: 0.4853544533252716
Hit 5, score: 0.6748016476631165
=================Hit 1=================
Scripting

 Arch API and FreeCAD Scripting Basics.

The Wall tool can be used in macros and from the Python console by using the following function:
```start_code
Wall = makeWall(baseobj=None, length=None, width=None, height=None, align="Center", face=None, name="Wall")
end_code```
Creates a  object from the given , which can be a Draft object, a Sketch, a face, or a solid.
 If no  is given, you can provide the numerical values for the ,  (thickness), and .
 If given,  can be used to give the index of a face from the underlying object, to build this wall on, instead of using the whole object.
  can be ,  

In [12]:
def retrieve_with_score(queries: np.ndarray, y: np.ndarray, k: int):
    for i in range(len(queries)):
        queries[i] += " python code"

    query_vectors = embedding_model.encode(queries)

    retrieved_texts = []
    scores = []

    for i in range(len(queries)):
        hits = client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vectors[i],
            limit=k,
        )

        retrieved_text_batch = [point.payload['content'] for point in hits.points]
        retrieved_code_batch = [filter_code(text) for text in retrieved_text_batch]


        retrieved_vec = embedding_model.encode(retrieved_code_batch)
        y_vec = embedding_model.encode(y[i])

        scores_batch = [cosine_similarity(x_vec, y_vec) for x_vec in retrieved_vec]

        scores.append(scores_batch)
        retrieved_texts.append(retrieved_text_batch)

    return retrieved_texts, scores

In [13]:
queries = query_df["X"].values
y = query_df["Y"].values
texts, scores = retrieve_with_score(
    queries=queries,
    y=y,
    k=5
)

In [126]:
bm_texts, bm_scores = bm25_retrieve_with_score(
    queries=queries,
    y=y,
    k=5
)

In [15]:
# Qdrant
# MACS 0.5307003046751022
# MMCS 0.6132178441286087
# Hit rate cosine 0.604

print("Qdrant")
print("MACS", mean_average_cosine_similarity(scores))
print("MMCS", mean_max_cosine_similarity(scores))
print("Hit rate cosine", hit_rate_at_k(scores, 0.6, 5))

Qdrant
MACS 0.5281057646989824
MMCS 0.6029994456768036
Hit rate cosine 0.512


In [128]:
print("BM25")
print("MACS", mean_average_cosine_similarity(bm_scores))
print("MMCS", mean_max_cosine_similarity(bm_scores))

BM25
MACS 0.46777109932899463
MMCS 0.5069063054323196
